In [40]:
import os
import boto3
import numpy as np
import pandas as pd
import tempfile
import keras
from huggingface_hub import hf_hub_download
from tornet.data.loader import read_file
import xarray as xr

print("numpy:", np.__version__)

# Setup S3
s3_client = boto3.client('s3')
bucket_name = 'ml-cloud-project-data'

# Download CSV from S3
s3_client.download_file(bucket_name, 'choo_choo_data_1.csv', 'choo_choo_data_1.csv')
df = pd.read_csv('choo_choo_data_1.csv')
print(f"Loaded {len(df)} rows")
print(df.head(2))

# Load pretrained model
model_file = hf_hub_download(
    repo_id="tornet-ml/tornado_detector_baseline_v1",
    filename="tornado_detector_baseline.keras"
)
cnn = keras.models.load_model(model_file, compile=False)
print("Model inputs:", list(cnn.input.keys()))

# Run inference
variables = ['DBZ', 'VEL', 'KDP', 'RHOHV', 'ZDR', 'WIDTH']
results = []

def make_coordinates(filepath, frame_idx):
    """Build the coordinates input the model expects from az/range dims"""
    ds = xr.open_dataset(filepath)
    az = ds['azimuth'].values        # (120,)
    rng = ds['range'].values         # (240,)
    
    # Create meshgrid and stack into coordinate array
    az_grid, rng_grid = np.meshgrid(az, rng, indexing='ij')  # (120, 240)
    
    # Normalize to roughly [-1, 1] range
    az_norm = az_grid / 360.0
    rng_norm = rng_grid / 150000.0  # max range ~150km
    
    # Stack into (1, 120, 240, 2)
    coords = np.stack([az_norm, rng_norm], axis=-1)[np.newaxis].astype(np.float32)
    ds.close()
    return coords

for k, v in cnn.input.items():
    print(k, v.shape)

for _, row in df.iterrows():
    s3_key = row['filepath'].replace(f's3://{bucket_name}/', '')
    frame_idx = row['frame_idx']
    
    try:
        with tempfile.NamedTemporaryFile(suffix='.nc', delete=True) as tmp:
            s3_client.download_file(bucket_name, s3_key, tmp.name)
            data = read_file(tmp.name, variables=variables, n_frames=4)
            coords = make_coordinates(tmp.name, frame_idx)
        
        xin = {k: data[k][[frame_idx]] for k in cnn.input if k in data}
        xin['coordinates'] = coords  # add coordinates
        
        logit = cnn.predict(xin, verbose=0)
        prob = float(1 / (1 + np.exp(-logit[0, 0])))
        
        results.append({
            'filepath': row['filepath'],
            'event_id': row['event_id'],
            'frame_idx': frame_idx,
            'label': row['label'],
            'tornado_prob': prob
        })
        print(f"✓ {row['event_id']} frame {frame_idx} → prob={prob:.3f}")

    except Exception as e:
        print(f"✗ Failed {row['event_id']} frame {frame_idx}: {e}")
        continue

results_df = pd.DataFrame(results)
results_df.to_csv('pretrained_inference_results.csv', index=False)

# Upload results back to S3
s3_client.upload_file('pretrained_inference_results.csv', bucket_name, 'pretrained_inference_results.csv')
print(f"\nDone. {len(results_df)}/{len(df)} succeeded.")

numpy: 1.26.4
Loaded 231 rows
                                            filepath  \
0  s3://ml-cloud-project-data/tornet_2014/train/2...   
1  s3://ml-cloud-project-data/tornet_2014/train/2...   

                           event_id  frame_idx         label  
0  TOR_140404_020014_KPAH_504204_P0          0  tornado_hook  
1  TOR_140404_020014_KPAH_504204_P0          1  tornado_hook  
Model inputs: ['DBZ', 'VEL', 'KDP', 'RHOHV', 'ZDR', 'WIDTH', 'range_folded_mask', 'coordinates']
DBZ (None, None, None, 2)
VEL (None, None, None, 2)
KDP (None, None, None, 2)
RHOHV (None, None, None, 2)
ZDR (None, None, None, 2)
WIDTH (None, None, None, 2)
range_folded_mask (None, None, None, 2)
coordinates (None, None, None, 2)
✓ TOR_140404_020014_KPAH_504204_P0 frame 0 → prob=0.006
✓ TOR_140404_020014_KPAH_504204_P0 frame 1 → prob=0.003
✓ TOR_140404_020014_KPAH_504204_P0 frame 2 → prob=0.035
✓ TOR_160523_003937_KMAF_631193_Y0 frame 0 → prob=0.197
✓ TOR_160523_003937_KMAF_631193_Y0 frame 1 → prob=0.235
✓

In [27]:
# Keep only tornado cases
tornado_df = df[df['label'].isin(['tornado_hook', 'tornado_no_hook'])].copy()

# Convert to binary labels
tornado_df['binary_label'] = tornado_df['label'].map({
    'tornado_no_hook': 0,
    'tornado_hook': 1
})

print(tornado_df['binary_label'].value_counts())

binary_label
0    55
1    52
Name: count, dtype: int64


In [28]:
X_data = []
coord_data = []
y_data = []

for _, row in tornado_df.iterrows():
    s3_key = row['filepath'].replace(f's3://{bucket_name}/', '')
    frame_idx = row['frame_idx']

    try:
        with tempfile.NamedTemporaryFile(suffix='.nc', delete=True) as tmp:
            s3_client.download_file(bucket_name, s3_key, tmp.name)
            data = read_file(tmp.name, variables=variables, n_frames=4)
            coords = make_coordinates(tmp.name, frame_idx)

        # Build model input dictionary
        xin = {k: data[k][[frame_idx]] for k in cnn.input if k in data}
        xin['coordinates'] = coords

        # Remove batch dimension before storing
        X_data.append({k: xin[k][0] for k in xin})
        y_data.append(row['binary_label'])

    except Exception as e:
        print("Failed:", e)
        continue

In [29]:
# Stack each input separately
X_stacked = {}
for key in X_data[0].keys():
    X_stacked[key] = np.stack([x[key] for x in X_data])

y_data = np.array(y_data)

display(X_stacked)

{'DBZ': array([[[[-7.5,  4.5],
          [-3.5, -1. ],
          [-1.5, -2. ],
          ...,
          [36.5, 29. ],
          [31.5, 30. ],
          [32.5, 30. ]],
 
         [[-8. , -1. ],
          [-5.5, -1. ],
          [-4. , -1.5],
          ...,
          [35.5, 32.5],
          [36.5, 29.5],
          [33. , 29. ]],
 
         [[-3.5, 15.5],
          [-3.5,  2.5],
          [-3.5,  8.5],
          ...,
          [35.5, 32.5],
          [34.5, 33.5],
          [37.5, 30.5]],
 
         ...,
 
         [[-0.5, -7.5],
          [ 2. ,  nan],
          [ 3.5, -2. ],
          ...,
          [34.5, 30. ],
          [35.5, 30.5],
          [34.5, 34.5]],
 
         [[-7. , -4.5],
          [15. , -3.5],
          [-0.5, -3. ],
          ...,
          [33.5, 31.5],
          [38.5, 36.5],
          [37.5, 31. ]],
 
         [[-1. , -6.5],
          [-2.5,  nan],
          [ 3.5, 13.5],
          ...,
          [36. , 42. ],
          [41. , 40.5],
          [41. , 40.5]]],
 
 
  

In [30]:
from sklearn.model_selection import train_test_split

X_train = {}
X_val = {}

for key in X_stacked:
    X_train[key], X_val[key], y_train, y_val = train_test_split(
        X_stacked[key], y_data,
        test_size=0.2,
        stratify=y_data,
        random_state=42
    )

In [31]:
indices = np.arange(len(y_data))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=y_data,
    random_state=42
)

X_train = {k: v[train_idx] for k, v in X_stacked.items()}
X_val   = {k: v[val_idx]   for k, v in X_stacked.items()}

y_train = y_data[train_idx]
y_val   = y_data[val_idx]

In [32]:
feature_extractor = keras.Model(
    inputs=cnn.input,
    outputs=cnn.layers[-2].output
)

# Freeze most layers
for layer in feature_extractor.layers[:-15]:
    layer.trainable = False

# Unfreeze last 15 layers
for layer in feature_extractor.layers[-15:]:
    layer.trainable = True

x = feature_extractor.output

# ADD THIS:
x = keras.layers.GlobalAveragePooling2D()(x)

x = keras.layers.Dense(64, activation='relu')(x)
x = keras.layers.Dropout(0.3)(x)
output = keras.layers.Dense(2, activation='softmax')(x)

model = keras.Model(
    inputs=feature_extractor.input,
    outputs=output
)

model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [33]:
for i, layer in enumerate(cnn.layers):
    print(i, layer.name, layer.output.shape)

feature_extractor = keras.Model(
    inputs=cnn.input,
    outputs=cnn.get_layer('block5_conv3').output  # replace with your actual layer name
)

# 2. Unfreeze more layers + raise LR slightly
for layer in feature_extractor.layers[:-8]:
    layer.trainable = False
for layer in feature_extractor.layers[-8:]:
    layer.trainable = True

# 3. Bigger head, stronger regularization
x = feature_extractor.output
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.5)(x)
output = keras.layers.Dense(2, activation='softmax')(x)

model = keras.Model(inputs=feature_extractor.input, outputs=output)

model.compile(
    optimizer=keras.optimizers.Adam(3e-5),  # slightly higher
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 4. Add class weights + early stopping
from sklearn.utils.class_weight import compute_class_weight

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(weights))

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=40,
    batch_size=8,
    class_weight=class_weight,
    callbacks=[early_stop]
)

0 DBZ (None, None, None, 2)
1 VEL (None, None, None, 2)
2 KDP (None, None, None, 2)
3 RHOHV (None, None, None, 2)
4 ZDR (None, None, None, 2)
5 WIDTH (None, None, None, 2)
6 Normalize_DBZ (None, None, None, 2)
7 Normalize_VEL (None, None, None, 2)
8 Normalize_KDP (None, None, None, 2)
9 Normalize_RHOHV (None, None, None, 2)
10 Normalize_ZDR (None, None, None, 2)
11 Normalize_WIDTH (None, None, None, 2)
12 Concatenate1 (None, None, None, 12)
13 range_folded_mask (None, None, None, 2)
14 Concatenate2 (None, None, None, 14)
15 coordinates (None, None, None, 2)


AttributeError: 'list' object has no attribute 'shape'

In [34]:
print(type(cnn))
cnn.summary()

<class 'keras.src.models.functional.Functional'>


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ DBZ (InputLayer)    │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ VEL (InputLayer)    │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ KDP (InputLayer)    │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ RHOHV (InputLayer)  │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ZDR (InputLayer)    │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ WIDTH (InputLayer)  │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_DBZ       │ (None, None,      │          0 │ DBZ[0][0]         │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_VEL       │ (None, None,      │          0 │ VEL[0][0]         │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_KDP       │ (None, None,      │          0 │ KDP[0][0]         │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_RHOHV     │ (None, None,      │          0 │ RHOHV[0][0]       │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_ZDR       │ (None, None,      │          0 │ ZDR[0][0]         │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_WIDTH     │ (None, None,      │          0 │ WIDTH[0][0]       │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Concatenate1        │ (None, None,      │          0 │ Normalize_DBZ[0]… │
│ (Concatenate)       │ None, 12)         │            │ Normalize_VEL[0]… │
│                     │                   │            │ Normalize_KDP[0]… │
│                     │                   │            │ Normalize_RHOHV[… │
│                     │                   │            │ Normalize_ZDR[0]… │
│                     │                   │            │ Normalize_WIDTH[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ isnan_4 (Isnan)     │ (None, None,      │          0 │ Concatenate1[0][… │
│                     │ None, 12)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ where_4 (Where)     │ (None, None,      │          0 │ isnan_4[0][0],    │
│                     │ None, 12)         │            │ Concatenate1[0][… │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,665,409 (17.80 MB)

 Trainable params: 4,508,737 (17.20 MB)

 Non-trainable params: 156,672 (612.00 KB)

In [24]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=8
)

model.evaluate(X_val, y_val)

from sklearn.metrics import confusion_matrix

y_pred = np.argmax(model.predict(X_val), axis=1)
print(confusion_matrix(y_val, y_pred))

Epoch 1/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 78s 4s/step - accuracy: 0.5153 - loss: 1.4553 - val_accuracy: 0.5000 - val_loss: 1.1326
Epoch 2/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.4524 - loss: 1.2355 - val_accuracy: 0.5000 - val_loss: 1.0325
Epoch 3/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 43s 4s/step - accuracy: 0.4515 - loss: 1.0264 - val_accuracy: 0.5000 - val_loss: 0.9606
Epoch 4/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 44s 4s/step - accuracy: 0.6053 - loss: 0.8755 - val_accuracy: 0.5000 - val_loss: 0.9145
Epoch 5/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 44s 4s/step - accuracy: 0.5236 - loss: 0.8859 - val_accuracy: 0.5000 - val_loss: 0.8746
Epoch 6/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 50s 5s/step - accuracy: 0.4512 - loss: 1.0092 - val_accuracy: 0.5000 - val_loss: 0.8405
Epoch 7/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 49s 5s/step - accuracy: 0.4972 - loss: 0.9634 - val_accuracy: 0.5000 - val_loss: 0.8139
Epoch 8/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 47s 4s/step - accuracy: 0.5302 - loss: 0.8017 - val_accuracy: 0.5000 - val_loss:

In [16]:
print("Train hook fraction:", np.mean(y_train))
print("Val hook fraction:", np.mean(y_val))

Train hook fraction: 0.4823529411764706
Val hook fraction: 0.5


## NEW TRY

In [36]:
import tensorflow as tf
from tensorflow import keras

# ── Inputs: only DBZ and VEL (2 channels each) ──────────────────────────────
dbz_input = keras.Input(shape=(None, None, 2), name='DBZ')
vel_input  = keras.Input(shape=(None, None, 2), name='VEL')

# ── Normalize ────────────────────────────────────────────────────────────────
dbz_norm = keras.layers.Normalization(name='Normalize_DBZ')(dbz_input)
vel_norm  = keras.layers.Normalization(name='Normalize_VEL')(vel_input)

# ── Concatenate (4 channels total instead of 14) ─────────────────────────────
x = keras.layers.Concatenate(name='Concatenate1')([dbz_norm, vel_norm])

# ── Replace NaNs with 0 ──────────────────────────────────────────────────────
x = keras.layers.ZeroPadding2D(padding=0)(x)  # no-op, just skip the nan line

# ── Conv block 1: 48 filters ─────────────────────────────────────────────────
x = keras.layers.Conv2D(48, 3, padding='same', activation='relu')(x)
x = keras.layers.Conv2D(48, 3, padding='same', activation='relu')(x)
x = keras.layers.MaxPooling2D(2)(x)
x = keras.layers.Dropout(0.2)(x)

# ── Conv block 2: 96 filters ─────────────────────────────────────────────────
x = keras.layers.Conv2D(96, 3, padding='same', activation='relu')(x)
x = keras.layers.Conv2D(96, 3, padding='same', activation='relu')(x)
x = keras.layers.MaxPooling2D(2)(x)
x = keras.layers.Dropout(0.2)(x)

# ── Conv block 3: 192 filters ────────────────────────────────────────────────
x = keras.layers.Conv2D(192, 3, padding='same', activation='relu')(x)
x = keras.layers.Conv2D(192, 3, padding='same', activation='relu')(x)
x = keras.layers.Conv2D(192, 3, padding='same', activation='relu')(x)
x = keras.layers.MaxPooling2D(2)(x)
x = keras.layers.Dropout(0.2)(x)

# ── Conv block 4: 384 filters ────────────────────────────────────────────────
x = keras.layers.Conv2D(384, 3, padding='same', activation='relu')(x)
x = keras.layers.Conv2D(384, 3, padding='same', activation='relu')(x)
x = keras.layers.Conv2D(384, 3, padding='same', activation='relu')(x)
x = keras.layers.MaxPooling2D(2)(x)
x = keras.layers.Dropout(0.2)(x)

# ── Classification head ───────────────────────────────────────────────────────
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Dropout(0.4)(x)
output = keras.layers.Dense(1, activation='sigmoid', name='hook')(x)

model = keras.Model(inputs=[dbz_input, vel_input], outputs=output)

model.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',    # sigmoid + binary is better than softmax + sparse_categorical for 2-class
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

model.summary()

Model: "functional_21"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ DBZ (InputLayer)    │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ VEL (InputLayer)    │ (None, None,      │          0 │ -                 │
│                     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_DBZ       │ (None, None,      │          5 │ DBZ[0][0]         │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Normalize_VEL       │ (None, None,      │          5 │ VEL[0][0]         │
│ (Normalization)     │ None, 2)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Concatenate1        │ (None, None,      │          0 │ Normalize_DBZ[0]… │
│ (Concatenate)       │ None, 4)          │            │ Normalize_VEL[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, None,      │          0 │ Concatenate1[0][… │
│ (ZeroPadding2D)     │ None, 4)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, None,      │      1,776 │ zero_padding2d[0… │
│                     │ None, 48)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, None,      │     20,784 │ conv2d_30[0][0]   │
│                     │ None, 48)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, None,      │          0 │ conv2d_31[0][0]   │
│ (MaxPooling2D)      │ None, 48)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, None,      │          0 │ max_pooling2d[0]… │
│                     │ None, 48)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, None,      │     41,568 │ dropout_5[0][0]   │
│                     │ None, 96)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, None,      │     83,040 │ conv2d_32[0][0]   │
│                     │ None, 96)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, None,      │          0 │ conv2d_33[0][0]   │
│ (MaxPooling2D)      │ None, 96)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, None,      │          0 │ max_pooling2d_1[… │
│                     │ None, 96)         │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, None,      │    166,080 │ dropout_6[0][0]   │
│                     │ None, 192)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, None,      │    331,968 │ conv2d_34[0][0]   │
│                     │ None, 192)        │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_36 (Conv2D)  │ (None, None,      │    331,968 │ conv2d_35[0][0] 

 Total params: 4,346,027 (16.58 MB)

 Trainable params: 4,345,761 (16.58 MB)

 Non-trainable params: 266 (1.05 KB)

In [38]:
X_train_dbz = X_train['DBZ']
X_train_vel  = X_train['VEL']
X_val_dbz   = X_val['DBZ']
X_val_vel    = X_val['VEL']

# Handle NaNs before training
X_train_dbz = np.nan_to_num(X_train_dbz, nan=0.0)
X_train_vel  = np.nan_to_num(X_train_vel,  nan=0.0)
X_val_dbz   = np.nan_to_num(X_val_dbz,   nan=0.0)
X_val_vel   = np.nan_to_num(X_val_vel,   nan=0.0)

In [39]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = {0: weights[0], 1: weights[1]}

early_stop = keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=7, restore_best_weights=True
)

history = model.fit(
    {'DBZ': X_train_dbz, 'VEL': X_train_vel},  # separate arrays per input
    y_train,
    validation_data=(
        {'DBZ': X_val_dbz, 'VEL': X_val_vel},
        y_val
    ),
    epochs=50,
    batch_size=8,
    class_weight=class_weight,
    callbacks=[early_stop]
)

Epoch 1/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 74s 6s/step - accuracy: 0.3202 - auc: 0.3169 - loss: 1.0374 - val_accuracy: 0.5000 - val_auc: 0.6488 - val_loss: 0.7305
Epoch 2/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 55s 5s/step - accuracy: 0.5277 - auc: 0.5095 - loss: 0.7629 - val_accuracy: 0.5000 - val_auc: 0.7025 - val_loss: 0.6827
Epoch 3/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 57s 5s/step - accuracy: 0.3827 - auc: 0.3597 - loss: 0.8891 - val_accuracy: 0.5909 - val_auc: 0.5620 - val_loss: 0.6865
Epoch 4/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 62s 6s/step - accuracy: 0.5691 - auc: 0.6281 - loss: 0.7030 - val_accuracy: 0.5000 - val_auc: 0.6694 - val_loss: 0.7321
Epoch 5/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 83s 6s/step - accuracy: 0.5494 - auc: 0.5011 - loss: 0.7831 - val_accuracy: 0.5000 - val_auc: 0.6612 - val_loss: 0.7250
Epoch 6/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 65s 6s/step - accuracy: 0.6102 - auc: 0.6204 - loss: 0.6950 - val_accuracy: 0.5000 - val_auc: 0.6612 - val_loss: 0.6976
Epoch 7/50
11/11 ━━━━━━━━━━━━━━━━━━━━ 65s 6s/step - 